# LeetCode #8 — String to Integer (atoi): đọc chuỗi theo từng bước và chống tràn số

**Mức độ:** Medium · **Chủ đề:** String, Simulation, Math

Skeleton trên LeetCode:

```python
class Solution:
    def myAtoi(self, s: str) -> int:
```

Bài này không có thuật toán "khéo" nào. Cái khó là **làm đúng từng quy tắc theo đúng thứ tự** và **không bỏ sót trường hợp biên**. Đây là kỹ năng rất quan trọng: đọc kỹ đặc tả rồi biến nó thành code tuần tự, có một con trỏ `i` đi dọc chuỗi.

Bài này dùng lại kỹ thuật **chống tràn số trước khi nhân** từ bài #7 Reverse Integer.

---

## Sau notebook này, bạn sẽ nắm được

1. Cách tách một đặc tả dài thành **4 bước** xử lý tuần tự.
2. Kỹ thuật dùng **một con trỏ `i`** đi qua chuỗi, mỗi bước "ăn" một phần.
3. Cách đổi ký tự chữ số thành số: `ord(c) - ord('0')`, và `c.isdigit()`.
4. Cách **kẹp (clamp)** kết quả vào phạm vi 32-bit, và phát hiện tràn trước khi nó xảy ra.
5. Hơn 10 trường hợp biên dễ sai.
6. Cách nhìn bài toán như một **máy trạng thái (state machine)**.

# 1. Bài toán là gì?

Viết hàm `myAtoi(s)` đổi chuỗi `s` thành số nguyên 32-bit có dấu, theo đúng thứ tự:

1. **Khoảng trắng:** bỏ qua mọi dấu cách `' '` ở đầu.
2. **Dấu:** nếu ký tự tiếp theo là `'-'` hoặc `'+'`, ghi nhận dấu. Không có thì coi là dương.
3. **Đọc số:** đọc các chữ số liên tiếp cho tới khi gặp ký tự không phải chữ số hoặc hết chuỗi. Nếu không đọc được chữ số nào, kết quả là `0`.
4. **Kẹp phạm vi:** nếu số vượt `[-2^31, 2^31 - 1]`, kẹp về biên gần nhất: nhỏ hơn `-2^31` thì thành `-2^31`, lớn hơn `2^31 - 1` thì thành `2^31 - 1`.

**Constraints (theo đề):**

- `0 <= s.length <= 200`
- `s` gồm chữ cái tiếng Anh (hoa, thường), chữ số, `' '`, `'+'`, `'-'`, `'.'`.

---

## Ví dụ

| `s` | Kết quả | Giải thích |
|---|---|---|
| `"42"` | `42` | Chỉ có chữ số |
| `"   -042"` | `-42` | Bỏ 3 dấu cách, dấu `-`, đọc `042` |
| `"1337c0d3"` | `1337` | Dừng ở chữ `c` |
| `"0-1"` | `0` | Đọc `0`, gặp `-` thì dừng (dấu chỉ hợp lệ ở bước 2) |
| `"words and 987"` | `0` | Ký tự đầu là `w`, không phải dấu hay chữ số → dừng ngay |

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Chuỗi (string) trong Python
2. Số nguyên 32 bit, tràn số và tách chữ số

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Chuỗi (string) trong Python

### Chuỗi là bất biến

`str` **không sửa được** tại chỗ: `s[0] = "x"` gây lỗi. Mọi "sửa" đều tạo chuỗi mới. Vì vậy:

- Ghép nhiều mảnh: gom vào **list** rồi `"".join(list)` (`O(n)`), tránh `s += ...` trong vòng lặp (có thể `O(n²)`).
- Cần sửa từng ký tự: `chars = list(s)`, sửa, rồi `"".join(chars)`.

In [1]:
s_str = "hello"
try:
    s_str[0] = "H"
except TypeError as e:
    print("Sửa trực tiếp:", e)
chars_str = list(s_str); chars_str[0] = "H"
print("qua list:", "".join(chars_str))
manh = []
for i in range(5):
    manh.append(str(i))
print("join:", "-".join(manh))

Sửa trực tiếp: 'str' object does not support item assignment
qua list: Hello
join: 0-1-2-3-4


### Cắt chuỗi (slicing) và các thao tác hay dùng

| Biểu thức | Ý nghĩa |
|---|---|
| `s[i:j]` | ký tự từ `i` tới `j−1` (tạo chuỗi mới, `O(j−i)`) |
| `s[::-1]` | chuỗi đảo ngược |
| `s.split()` | tách theo khoảng trắng (bỏ khoảng trắng thừa) |
| `s.split(",")` | tách theo dấu phẩy (giữ phần rỗng) |
| `s.strip()` | bỏ khoảng trắng hai đầu |
| `s.startswith(p)`, `s.find(p)` | kiểm tra tiền tố, tìm vị trí |
| `c.isdigit()`, `c.isalpha()`, `c.isalnum()`, `c.isspace()` | phân loại ký tự |
| `c.lower()`, `c.upper()` | đổi hoa/thường |

In [2]:
t_str = "  Xin chào   thế giới  "
print(repr(t_str.split()), "|", repr(t_str.strip()), "|", repr("a,,b".split(",")))
print("đảo:", "abcdef"[::-1], "| cắt [1:4]:", "abcdef"[1:4], "| 'abc'.find('c') =", "abc".find("c"))
print([(c, c.isdigit(), c.isalpha()) for c in "a1 "])

['Xin', 'chào', 'thế', 'giới'] | 'Xin chào   thế giới' | ['a', '', 'b']
đảo: fedcba | cắt [1:4]: bcd | 'abc'.find('c') = 2
[('a', False, True), ('1', True, False), (' ', False, False)]


### Ký tự và mã số: `ord` / `chr`

Mỗi ký tự có một mã số (Unicode). `ord("a") = 97`, `chr(97) = "a"`. Mẹo: chữ số `c` → giá trị `ord(c) - ord("0")`; chữ thường `c` → chỉ số 0..25 là `ord(c) - ord("a")`.

In [3]:
print("ord('a') =", ord("a"), "| chr(98) =", chr(98))
print("'7' → 7:", ord("7") - ord("0"), "| 'd' → chỉ số:", ord("d") - ord("a"))
dem26 = [0] * 26
for c in "banana":
    dem26[ord(c) - ord("a")] += 1
print("đếm 26 chữ cái của 'banana' (a, b, n):", dem26[0], dem26[1], dem26[13])

ord('a') = 97 | chr(98) = b
'7' → 7: 7 | 'd' → chỉ số: 3
đếm 26 chữ cái của 'banana' (a, b, n): 3 1 2


### Gom nhóm ký tự liên tiếp

`itertools.groupby` gom các phần tử **liên tiếp bằng nhau** — tiện cho mã hóa độ dài loạt (run-length): `"aaabcc" → 3a 1b 2c`. Tự viết bằng hai con trỏ: `i` ở đầu nhóm, `j` chạy tới khi ký tự khác.

In [4]:
from itertools import groupby
print([(k, len(list(g))) for k, g in groupby("aaabcc")])

def run_length(s):
    out, i = [], 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        out.append(f"{j - i}{s[i]}")
        i = j
    return "".join(out)
print("run_length('aaabcc') =", run_length("aaabcc"))

[('a', 3), ('b', 1), ('c', 2)]
run_length('aaabcc') = 3a1b2c


Lỗi hay gặp: `split(" ")` sinh phần rỗng khi có nhiều dấu cách; so sánh chuỗi số `"10" < "9"` là **True** (so từ điển, không phải số); nối chuỗi trong vòng lặp lớn; quên chuỗi rỗng.

## 2.2 Số nguyên 32 bit, tràn số và tách chữ số

### Giới hạn 32 bit có dấu

Nhiều đề yêu cầu "môi trường chỉ lưu được số nguyên 32 bit có dấu":

```text
INT_MIN = -2^31     = -2147483648
INT_MAX =  2^31 - 1 =  2147483647
```

Vượt khỏi khoảng này gọi là **tràn số** (overflow). C/Java sẽ cho kết quả sai (quay vòng); Python thì số nguyên **không giới hạn**, nên ta phải **tự kiểm tra** để đúng yêu cầu đề.

In [5]:
INT_MIN_D, INT_MAX_D = -2 ** 31, 2 ** 31 - 1
print(INT_MIN_D, INT_MAX_D)
print("Python không tràn:", INT_MAX_D + 1, "| mô phỏng C (quay vòng):", ((INT_MAX_D + 1 + 2 ** 31) % 2 ** 32) - 2 ** 31)

def kep_32(x):
    return max(INT_MIN_D, min(INT_MAX_D, x))
print("kẹp 10^10 về:", kep_32(10 ** 10), "| kẹp -10^10:", kep_32(-10 ** 10))

-2147483648 2147483647
Python không tràn: 2147483648 | mô phỏng C (quay vòng): -2147483648
kẹp 10^10 về: 2147483647 | kẹp -10^10: -2147483648


### Tách và ghép chữ số

- Chữ số cuối: `x % 10`; bỏ chữ số cuối: `x // 10` (với `x >= 0`).
- Ghép thêm chữ số `d` vào bên phải: `x = x * 10 + d`.
- `divmod(x, 10)` trả cả hai cùng lúc.

**Cẩn thận với số âm:** trong Python `-123 // 10 = -13` và `-123 % 10 = 7` (làm tròn **xuống**). Nên xử lý **dấu riêng**, làm việc với `abs(x)`.

In [6]:
x_d = 12345
chu_so = []
while x_d > 0:
    x_d, d = divmod(x_d, 10)
    chu_so.append(d)
print("chữ số (từ phải sang):", chu_so)
y_d = 0
for d in [5, 4, 3, 2, 1]:
    y_d = y_d * 10 + d
print("ghép lại:", y_d)
print("số âm: -123 // 10 =", -123 // 10, "| -123 % 10 =", -123 % 10, "| int(-123 / 10) =", int(-123 / 10))

chữ số (từ phải sang): [5, 4, 3, 2, 1]
ghép lại: 54321
số âm: -123 // 10 = -13 | -123 % 10 = 7 | int(-123 / 10) = -12


### Kiểm tra tràn **trước khi** tràn

Khi không được dùng số lớn hơn 32 bit (như trong C), phải kiểm tra trước bước `x = x * 10 + d`:

```text
x * 10 + d > INT_MAX   ⟺   x > INT_MAX // 10   hoặc   (x == INT_MAX // 10 và d > 7)
```

(vì `INT_MAX = 214748364·10 + 7`). Trong Python có thể tính rồi so sánh, nhưng hiểu điều kiện này giúp viết đúng ở mọi ngôn ngữ.

In [7]:
def ghep_an_toan(chu_so_list):
    x = 0
    for d in chu_so_list:
        if x > INT_MAX_D // 10 or (x == INT_MAX_D // 10 and d > 7):
            return None                                   # sẽ tràn
        x = x * 10 + d
    return x

print(ghep_an_toan([2, 1, 4, 7, 4, 8, 3, 6, 4, 7]), ghep_an_toan([2, 1, 4, 7, 4, 8, 3, 6, 4, 8]))

2147483647 None


## 2.3 Áp dụng vào bài này

Bài là **đọc chuỗi theo từng bước** với con trỏ `i`: bỏ khoảng trắng đầu, đọc dấu, đọc các ký tự chữ số (`isdigit`, `ord(c) − ord('0')`) và **ghép chữ số** thành số. Kết quả phải **kẹp** vào khoảng 32 bit có dấu — chính là kiến thức tràn số ở trên.

# 3. Nền tảng: đi qua chuỗi bằng con trỏ `i`

Thay vì dùng `for c in s`, ta dùng chỉ số `i` để mỗi bước có thể **dừng giữa chừng** và bước sau **tiếp tục từ chỗ đó**.

```python
i = 0
while i < n and s[i] == ' ':   # bước 1: ăn hết dấu cách
    i += 1
# lúc này s[i] là ký tự đầu tiên không phải dấu cách (nếu i < n)
```

Luôn kiểm tra `i < n` **trước** khi đọc `s[i]`, nếu không sẽ bị `IndexError` khi chuỗi toàn dấu cách.

In [8]:
s = "   -042"
n = len(s)
i = 0
while i < n and s[i] == " ":
    i += 1
print("Sau bước 1, i =", i, "-> s[i] =", repr(s[i]))

s = "    "
n = len(s)
i = 0
while i < n and s[i] == " ":
    i += 1
print("Chuỗi toàn dấu cách: i =", i, "= n, không còn gì để đọc")

Sau bước 1, i = 3 -> s[i] = '-'
Chuỗi toàn dấu cách: i = 4 = n, không còn gì để đọc


## Đổi ký tự chữ số thành số

Mỗi ký tự có một mã số (`ord`). Các chữ số `'0'`..`'9'` có mã liên tiếp, nên:

```python
digit = ord(c) - ord('0')    # '7' -> 7
```

Kiểm tra một ký tự có phải chữ số: `'0' <= c <= '9'` hoặc `c.isdigit()`.

> Lưu ý: `isdigit()` cũng trả `True` cho một số ký tự Unicode như `'²'`. Theo constraints, `s` chỉ có ký tự ASCII nên ở bài này dùng gì cũng được; `'0' <= c <= '9'` là chặt chẽ nhất.

In [9]:
for c in "07a.":
    print(repr(c), "ord =", ord(c), "| là chữ số?", "0" <= c <= "9",
          "| digit =", ord(c) - ord("0") if "0" <= c <= "9" else None)

'0' ord = 48 | là chữ số? True | digit = 0
'7' ord = 55 | là chữ số? True | digit = 7
'a' ord = 97 | là chữ số? False | digit = None
'.' ord = 46 | là chữ số? False | digit = None


## Ghép chữ số thành số

Đọc từ trái sang phải, mỗi chữ số mới **đẩy** các chữ số cũ sang trái một hàng:

```text
value = value * 10 + digit
"042":  0 -> 0*10+4 = 4 -> 4*10+2 = 42
```

Số `0` ở đầu tự biến mất, không cần xử lý riêng.

In [10]:
value = 0
for c in "042":
    value = value * 10 + (ord(c) - ord("0"))
    print(f"đọc {c!r} -> value = {value}")

đọc '0' -> value = 0
đọc '4' -> value = 4
đọc '2' -> value = 42


# 4. Cách dễ nghĩ: cắt phần hợp lệ rồi dùng `int()`

1. `s.lstrip(' ')` bỏ dấu cách đầu.
2. Lấy dấu nếu có.
3. Lấy tiền tố toàn chữ số.
4. `int(...)`, gắn dấu, kẹp phạm vi.

In [11]:
INT_MAX = 2**31 - 1
INT_MIN = -2**31

def my_atoi_slice(s: str) -> int:
    s = s.lstrip(" ")
    sign = 1
    if s and s[0] in "+-":
        sign = -1 if s[0] == "-" else 1
        s = s[1:]

    digits = ""
    for c in s:
        if not ("0" <= c <= "9"):
            break
        digits += c

    if not digits:
        return 0
    value = sign * int(digits)
    return max(INT_MIN, min(INT_MAX, value))

for s in ["42", "   -042", "1337c0d3", "0-1", "words and 987", "-91283472332"]:
    print(f"{s!r:18} -> {my_atoi_slice(s)}")

'42'               -> 42
'   -042'          -> -42
'1337c0d3'         -> 1337
'0-1'              -> 0
'words and 987'    -> 0
'-91283472332'     -> -2147483648


## Vấn đề của cách này

- Đúng trong Python, nhưng `int(digits)` có thể tạo số **200 chữ số** — trong môi trường 32-bit, nó tràn trước khi ta kịp kẹp.
- Tạo nhiều chuỗi trung gian (`lstrip`, `s[1:]`, `digits += c`).
- Đề bài gốc là bài luyện "tự viết atoi" — dùng `int()` là né mất phần chính.

Lời giải chính sẽ đọc từng chữ số và **dừng ngay khi biết sẽ tràn**, giống bài #7.

# 5. Chống tràn: nhắc lại từ bài #7

Trước khi tính `value * 10 + digit`, kiểm tra nó có vượt `limit` không:

```python
if value > limit // 10 or (value == limit // 10 and digit > limit % 10):
    # tràn!
```

Với `limit = 2147483647`: `limit // 10 = 214748364`, `limit % 10 = 7`.

Khác bài #7: khi tràn, ta **không trả về 0** mà **kẹp** về `INT_MAX` (dương) hoặc `INT_MIN` (âm). Phía âm cho phép độ lớn `2^31`, nên dùng `limit = 2^31` khi dấu âm.

In [12]:
def overflows(value, digit, limit):
    return value > limit // 10 or (value == limit // 10 and digit > limit % 10)

print(overflows(214748364, 7, INT_MAX))     # False -> 2147483647 hợp lệ
print(overflows(214748364, 8, INT_MAX))     # True  -> dương, kẹp về INT_MAX
print(overflows(214748364, 8, 2**31))       # False -> âm, -2147483648 hợp lệ
print(overflows(214748364, 9, 2**31))       # True  -> âm, kẹp về INT_MIN

False
True
False
True


# 6. Lời giải chính: 4 bước với một con trỏ

```text
i = 0
Bước 1: while s[i] == ' ': i += 1
Bước 2: if s[i] in '+-': ghi dấu, i += 1        (chỉ MỘT dấu)
Bước 3: while s[i] là chữ số: kiểm tra tràn, value = value*10 + digit, i += 1
Bước 4: return sign * value   (kẹp đã làm trong bước 3)
```

In [13]:
def my_atoi(s: str) -> int:
    n = len(s)
    i = 0

    # Bước 1: bỏ dấu cách đầu
    while i < n and s[i] == " ":
        i += 1

    # Bước 2: dấu
    sign = 1
    if i < n and s[i] in "+-":
        sign = -1 if s[i] == "-" else 1
        i += 1

    # Bước 3 + 4: đọc chữ số, chống tràn bằng cách kẹp
    limit = INT_MAX if sign == 1 else INT_MAX + 1
    value = 0
    while i < n and "0" <= s[i] <= "9":
        digit = ord(s[i]) - ord("0")
        if value > limit // 10 or (value == limit // 10 and digit > limit % 10):
            return INT_MAX if sign == 1 else INT_MIN
        value = value * 10 + digit
        i += 1

    return sign * value

for s in ["42", "   -042", "1337c0d3", "0-1", "words and 987", "-91283472332"]:
    print(f"{s!r:18} -> {my_atoi(s)}")

'42'               -> 42
'   -042'          -> -42
'1337c0d3'         -> 1337
'0-1'              -> 0
'words and 987'    -> 0
'-91283472332'     -> -2147483648


# 7. Mô phỏng từng bước

Xét `s = "  -0012a42"`:

| Bước | i | Ký tự | Hành động | sign | value |
|---|---|---|---|---|---|
| 1 | 0, 1 | `' '`, `' '` | bỏ qua | | |
| 2 | 2 | `'-'` | ghi dấu âm | -1 | 0 |
| 3 | 3 | `'0'` | value = 0·10 + 0 | -1 | 0 |
| 3 | 4 | `'0'` | value = 0·10 + 0 | -1 | 0 |
| 3 | 5 | `'1'` | value = 0·10 + 1 | -1 | 1 |
| 3 | 6 | `'2'` | value = 1·10 + 2 | -1 | 12 |
| 3 | 7 | `'a'` | không phải chữ số → dừng | | |
| 4 | | | trả về -1 · 12 | | **-12** |

Phần `"42"` sau chữ `a` bị bỏ qua hoàn toàn.

In [14]:
def trace_atoi(s: str) -> int:
    n = len(s)
    i = 0
    print(f"s = {s!r}")
    while i < n and s[i] == " ":
        i += 1
    print(f"  Bước 1: bỏ {i} dấu cách, i = {i}")

    sign = 1
    if i < n and s[i] in "+-":
        sign = -1 if s[i] == "-" else 1
        print(f"  Bước 2: gặp {s[i]!r}, sign = {sign}")
        i += 1
    else:
        print("  Bước 2: không có dấu, sign = 1")

    limit = INT_MAX if sign == 1 else INT_MAX + 1
    value = 0
    while i < n and "0" <= s[i] <= "9":
        digit = ord(s[i]) - ord("0")
        if value > limit // 10 or (value == limit // 10 and digit > limit % 10):
            clamped = INT_MAX if sign == 1 else INT_MIN
            print(f"  Bước 3: i = {i}, digit = {digit}, value = {value} -> sẽ tràn, kẹp về {clamped}")
            return clamped
        value = value * 10 + digit
        print(f"  Bước 3: i = {i}, digit = {digit}, value = {value}")
        i += 1

    stop = repr(s[i]) if i < n else "hết chuỗi"
    print(f"  Dừng đọc vì: {stop}")
    print(f"  Bước 4: kết quả = {sign} * {value} = {sign * value}")
    return sign * value

trace_atoi("  -0012a42")
print()
trace_atoi("2147483648")

s = '  -0012a42'
  Bước 1: bỏ 2 dấu cách, i = 2
  Bước 2: gặp '-', sign = -1
  Bước 3: i = 3, digit = 0, value = 0
  Bước 3: i = 4, digit = 0, value = 0
  Bước 3: i = 5, digit = 1, value = 1
  Bước 3: i = 6, digit = 2, value = 12
  Dừng đọc vì: 'a'
  Bước 4: kết quả = -1 * 12 = -12

s = '2147483648'
  Bước 1: bỏ 0 dấu cách, i = 0
  Bước 2: không có dấu, sign = 1
  Bước 3: i = 0, digit = 2, value = 2
  Bước 3: i = 1, digit = 1, value = 21
  Bước 3: i = 2, digit = 4, value = 214
  Bước 3: i = 3, digit = 7, value = 2147
  Bước 3: i = 4, digit = 4, value = 21474
  Bước 3: i = 5, digit = 8, value = 214748
  Bước 3: i = 6, digit = 3, value = 2147483
  Bước 3: i = 7, digit = 6, value = 21474836
  Bước 3: i = 8, digit = 4, value = 214748364
  Bước 3: i = 9, digit = 8, value = 214748364 -> sẽ tràn, kẹp về 2147483647


2147483647

# 8. Phiên bản đúng format LeetCode

In [15]:
class Solution:
    def myAtoi(self, s: str) -> int:
        INT_MAX, INT_MIN = 2**31 - 1, -2**31
        n = len(s)
        i = 0

        while i < n and s[i] == " ":
            i += 1

        sign = 1
        if i < n and s[i] in "+-":
            sign = -1 if s[i] == "-" else 1
            i += 1

        limit = INT_MAX if sign == 1 else INT_MAX + 1
        value = 0
        while i < n and "0" <= s[i] <= "9":
            digit = ord(s[i]) - ord("0")
            if value > limit // 10 or (value == limit // 10 and digit > limit % 10):
                return INT_MAX if sign == 1 else INT_MIN
            value = value * 10 + digit
            i += 1

        return sign * value

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<34} PASS  {actual}")


sol = Solution()
check("Ví dụ 1: '42'", sol.myAtoi("42"), 42)
check("Ví dụ 2: '   -042'", sol.myAtoi("   -042"), -42)
check("Ví dụ 3: '1337c0d3'", sol.myAtoi("1337c0d3"), 1337)
check("Ví dụ 4: '0-1'", sol.myAtoi("0-1"), 0)
check("Ví dụ 5: 'words and 987'", sol.myAtoi("words and 987"), 0)
check("Chuỗi rỗng", sol.myAtoi(""), 0)
check("Toàn dấu cách", sol.myAtoi("    "), 0)
check("Chỉ có dấu", sol.myAtoi("-"), 0)
check("Hai dấu liền nhau", sol.myAtoi("+-12"), 0)
check("Dấu cách sau dấu", sol.myAtoi("- 12"), 0)
check("Dấu chấm thập phân", sol.myAtoi("3.14159"), 3)
check("Tràn âm", sol.myAtoi("-91283472332"), -2147483648)
check("Tràn dương", sol.myAtoi("2147483648"), 2147483647)
check("Vừa đúng INT_MIN", sol.myAtoi("-2147483648"), -2147483648)
check("Vừa đúng INT_MAX", sol.myAtoi("2147483647"), 2147483647)
check("Rất nhiều số 0 đầu", sol.myAtoi("0000000000012345678"), 12345678)

Ví dụ 1: '42'                      PASS  42
Ví dụ 2: '   -042'                 PASS  -42
Ví dụ 3: '1337c0d3'                PASS  1337
Ví dụ 4: '0-1'                     PASS  0
Ví dụ 5: 'words and 987'           PASS  0
Chuỗi rỗng                         PASS  0
Toàn dấu cách                      PASS  0
Chỉ có dấu                         PASS  0
Hai dấu liền nhau                  PASS  0
Dấu cách sau dấu                   PASS  0
Dấu chấm thập phân                 PASS  3
Tràn âm                            PASS  -2147483648
Tràn dương                         PASS  2147483647
Vừa đúng INT_MIN                   PASS  -2147483648
Vừa đúng INT_MAX                   PASS  2147483647
Rất nhiều số 0 đầu                 PASS  12345678


## Đối chiếu với cách cắt chuỗi + `int()`

In [17]:
import random

random.seed(8)
alphabet = "  +-0123456789a."
for _ in range(20000):
    s = "".join(random.choice(alphabet) for _ in range(random.randint(0, 15)))
    assert sol.myAtoi(s) == my_atoi_slice(s), s
for _ in range(2000):
    s = random.choice(["", "-", "+", " "]) + "".join(random.choice("0123456789") for _ in range(random.randint(8, 30)))
    assert sol.myAtoi(s) == my_atoi_slice(s), s
print("22000 test ngẫu nhiên: hai cách khớp nhau")

22000 test ngẫu nhiên: hai cách khớp nhau


# 9. Vì sao đúng? Độ phức tạp?

**Đúng:** code là bản dịch trực tiếp 4 bước của đề, theo đúng thứ tự. Con trỏ `i` chỉ đi tới, nên mỗi bước bắt đầu đúng nơi bước trước dừng. Phép kiểm tra tràn đúng **khi và chỉ khi** `value * 10 + digit > limit` (đã chứng minh ở bài #7); và `value` chỉ tăng, nên một khi vượt giới hạn thì kết quả chắc chắn bị kẹp.

**Độ phức tạp** (gọi `n = len(s)`):

```text
Time:  O(n)   — i đi qua mỗi ký tự tối đa một lần (thực tế dừng sớm khi tràn: tối đa ~10 chữ số có nghĩa)
Space: O(1)   — chỉ vài biến
```

Cách cắt chuỗi cũng `O(n)` thời gian nhưng `O(n)` bộ nhớ và tạo ra số có thể rất lớn.

# 10. Những lỗi dễ mắc

## Lỗi 1: Chấp nhận nhiều dấu, hoặc dấu ở giữa

Dấu chỉ hợp lệ **đúng một lần**, ngay sau khoảng trắng. `"+-12"` → `0`, `"0-1"` → `0`.

In [18]:
def wrong_many_signs(s: str) -> int:
    i, n, sign = 0, len(s), 1
    while i < n and s[i] == " ":
        i += 1
    while i < n and s[i] in "+-":          # SAI: while thay vì if
        if s[i] == "-":
            sign = -sign
        i += 1
    value = 0
    while i < n and "0" <= s[i] <= "9":
        value = value * 10 + ord(s[i]) - ord("0")
        i += 1
    return sign * value

print("Sai :", wrong_many_signs("+-12"))
print("Đúng:", my_atoi("+-12"))

Sai : -12
Đúng: 0


## Lỗi 2: Dùng `strip()` thay vì chỉ bỏ dấu cách **đầu**

`strip()` bỏ cả hai đầu — vô hại ở đây. Nhưng `s.split()` hay bỏ mọi khoảng trắng sẽ biến `"- 12"` thành `"-12"` → `-12`, trong khi đáp án đúng là `0`.

## Lỗi 3: Quên kiểm tra `i < n`

Với `""`, `"   "`, `"-"`: đọc `s[i]` khi `i == n` sẽ gây `IndexError`.

In [19]:
def wrong_no_bound_check(s: str) -> int:
    i = 0
    while s[i] == " ":                      # SAI: thiếu i < len(s)
        i += 1
    return i

try:
    wrong_no_bound_check("   ")
except IndexError as e:
    print("Lỗi:", type(e).__name__, "-", e)

Lỗi: IndexError - string index out of range


## Lỗi 4: Trả về 0 khi tràn (nhầm với bài #7)

Bài #7 trả `0`, bài này **kẹp** về `INT_MAX` / `INT_MIN`.

## Lỗi 5: Kẹp sai phía

`"-91283472332"` phải thành `-2147483648`, không phải `2147483647`. Nhớ dùng `sign` để chọn biên.

## Lỗi 6: Bỏ qua cả ký tự khác dấu cách ở đầu

Đề chỉ bỏ `' '`. Một chữ cái ở đầu (như `"words and 987"`) làm kết quả là `0` ngay.

# 11. Góc nhìn khác: máy trạng thái (state machine)

Ta có thể mô tả atoi bằng các **trạng thái** và **chuyển trạng thái** theo loại ký tự:

| Trạng thái \ Ký tự | dấu cách | `+` / `-` | chữ số | khác |
|---|---|---|---|---|
| `start` | `start` | `signed` | `number` | `end` |
| `signed` | `end` | `end` | `number` | `end` |
| `number` | `end` | `end` | `number` | `end` |
| `end` | `end` | `end` | `end` | `end` |

Cách này dài hơn cho bài này, nhưng rất mạnh khi quy tắc phức tạp hơn — ví dụ bài **#65 Valid Number** (Hard).

In [20]:
TRANSITIONS = {
    "start":  {"space": "start", "sign": "signed", "digit": "number", "other": "end"},
    "signed": {"space": "end",   "sign": "end",    "digit": "number", "other": "end"},
    "number": {"space": "end",   "sign": "end",    "digit": "number", "other": "end"},
    "end":    {"space": "end",   "sign": "end",    "digit": "end",    "other": "end"},
}

def char_kind(c):
    if c == " ":
        return "space"
    if c in "+-":
        return "sign"
    if "0" <= c <= "9":
        return "digit"
    return "other"

def my_atoi_fsm(s: str) -> int:
    state, sign, value = "start", 1, 0
    for c in s:
        state = TRANSITIONS[state][char_kind(c)]
        if state == "end":
            break
        if state == "signed":
            sign = -1 if c == "-" else 1
        elif state == "number":
            value = value * 10 + ord(c) - ord("0")
            if value > INT_MAX + 1:          # chặn sớm để value không phình to
                break
    return max(INT_MIN, min(INT_MAX, sign * value))

for _ in range(5000):
    s = "".join(random.choice(alphabet) for _ in range(random.randint(0, 15)))
    assert my_atoi_fsm(s) == sol.myAtoi(s), s
print("Máy trạng thái khớp lời giải chính trên 5000 test")

Máy trạng thái khớp lời giải chính trên 5000 test


# 12. Mẹo nhận diện pattern

**Bài "parse / đọc chuỗi theo đặc tả":**

- Đề liệt kê quy tắc theo **thứ tự** → viết code theo đúng thứ tự đó, mỗi quy tắc một đoạn.
- Dùng **một con trỏ `i`** đi tới; mỗi đoạn "ăn" một phần chuỗi.
- Trước mỗi lần đọc `s[i]`, kiểm tra `i < n`.
- Liệt kê test biên **trước khi code**: rỗng, toàn khoảng trắng, chỉ có dấu, dấu kép, tràn hai phía.

**Bài có "32-bit" + "clamp / overflow":** dùng mẫu kiểm tra `limit // 10`, `limit % 10`.

Các bài liên quan: **#7 Reverse Integer**, **#65 Valid Number**, **#224 Basic Calculator**, **#468 Validate IP Address**.

# 13. Template tư duy

```python
i, n = 0, len(s)
while i < n and s[i] == ' ':           # 1. khoảng trắng
    i += 1

sign = 1
if i < n and s[i] in '+-':             # 2. đúng MỘT dấu
    sign = -1 if s[i] == '-' else 1
    i += 1

limit = INT_MAX if sign == 1 else INT_MAX + 1
value = 0
while i < n and '0' <= s[i] <= '9':    # 3. chữ số
    d = ord(s[i]) - ord('0')
    if value > limit // 10 or (value == limit // 10 and d > limit % 10):
        return INT_MAX if sign == 1 else INT_MIN   # 4. kẹp
    value = value * 10 + d
    i += 1
return sign * value
```

# 14. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Không chạy code. Kết quả `myAtoi` của mỗi chuỗi sau là gì? Ở mỗi chuỗi, việc đọc dừng ở ký tự nào?

```text
"   +0 123"
"-5-"
"  0000000000000000000000000001"
".1"
"-2147483649"
```

## Bài 2 — Bỏ khoảng trắng đầu

Viết hàm trả về chỉ số của ký tự đầu tiên không phải dấu cách (trả `len(s)` nếu toàn dấu cách).

```python
skip_spaces("   ab") -> 3
skip_spaces("ab")    -> 0
skip_spaces("   ")   -> 3
```

In [21]:
def exercise_2_skip_spaces(s: str) -> int:
    # TODO: dùng while với điều kiện i < len(s) and s[i] == ' '
    pass


# assert exercise_2_skip_spaces("   ab") == 3
# assert exercise_2_skip_spaces("ab") == 0
# assert exercise_2_skip_spaces("   ") == 3

## Bài 3 — Điền chỗ trống: lời giải chính

In [22]:
def exercise_3_my_atoi(s: str) -> int:
    INT_MAX, INT_MIN = 2**31 - 1, -2**31
    n, i = len(s), 0
    # TODO 1: bỏ dấu cách đầu
    # TODO 2: đọc tối đa MỘT dấu
    # TODO 3: đọc chữ số, kiểm tra tràn trước khi nhân, kẹp nếu tràn
    # TODO 4: trả về sign * value
    pass


# assert exercise_3_my_atoi("   -042") == -42
# assert exercise_3_my_atoi("+-12") == 0
# assert exercise_3_my_atoi("-91283472332") == -2147483648
# assert exercise_3_my_atoi("") == 0

## Bài 4 — Biến thể: atoi 8-bit

Giống `myAtoi`, nhưng phạm vi là số nguyên **8-bit có dấu** `[-128, 127]`. Hãy tổng quát hóa: viết hàm nhận thêm tham số `bits`.

```python
atoi_bits("200", 8)   -> 127
atoi_bits("-128", 8)  -> -128
atoi_bits("-129", 8)  -> -128
atoi_bits("42", 32)   -> 42
```

In [23]:
def exercise_4_atoi_bits(s: str, bits: int) -> int:
    # TODO
    pass


# assert exercise_4_atoi_bits("200", 8) == 127
# assert exercise_4_atoi_bits("-128", 8) == -128
# assert exercise_4_atoi_bits("-129", 8) == -128
# assert exercise_4_atoi_bits("42", 32) == 42

## Bài 5 — Mini challenge: đọc tất cả số nguyên trong chuỗi

Viết hàm trả về **danh sách mọi số nguyên** xuất hiện trong chuỗi. Một số nguyên là một dãy chữ số liên tiếp, có thể có **một** dấu `-` đứng **ngay** trước. Không cần kẹp phạm vi.

```python
extract_ints("a12b-3c--4 5")  -> [12, -3, -4, 5]
extract_ints("no digits")     -> []
extract_ints("-")             -> []
```

Gợi ý nhanh: dùng con trỏ `i`; khi gặp chữ số (hoặc `-` ngay trước chữ số), đọc hết số rồi tiếp tục từ chỗ dừng.

In [24]:
def exercise_5_extract_ints(s: str):
    # TODO
    pass


# assert exercise_5_extract_ints("a12b-3c--4 5") == [12, -3, -4, 5]
# assert exercise_5_extract_ints("no digits") == []
# assert exercise_5_extract_ints("-") == []

# 15. Gợi ý

**Bài 1**
- Mức 1: đi đúng 4 bước, đừng "nhảy cóc".
- Mức 2: sau dấu, ký tự kế tiếp phải là chữ số, nếu không thì dừng.
- Mức 3: `"-2147483649"` vượt phía âm đúng 1 đơn vị.

**Bài 3**
- Mức 1: so với template mục 13.
- Mức 2: bước 2 dùng `if`, không dùng `while`.
- Mức 3: `limit = INT_MAX if sign == 1 else INT_MAX + 1`.

**Bài 4**
- Mức 1: với `bits`, `MAX = 2**(bits-1) - 1`, `MIN = -2**(bits-1)`.
- Mức 2: copy lời giải chính, thay hai hằng số.
- Mức 3: `limit = MAX if sign == 1 else MAX + 1`.

**Bài 5**
- Mức 1: vòng ngoài `while i < n`.
- Mức 2: số bắt đầu khi `s[i]` là chữ số, **hoặc** `s[i] == '-'` và `s[i+1]` là chữ số.
- Mức 3: sau khi đọc xong một số, **không** tăng `i` thêm — vòng đọc chữ số đã dừng đúng chỗ.

# 16. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

| Chuỗi | Kết quả | Dừng ở |
|---|---|---|
| `"   +0 123"` | `0` | dấu cách sau `0` |
| `"-5-"` | `-5` | dấu `-` thứ hai |
| `"  000…0001"` | `1` | hết chuỗi |
| `".1"` | `0` | `.` (không phải dấu, không phải chữ số) |
| `"-2147483649"` | `-2147483648` | chữ số cuối (phát hiện tràn, kẹp) |

In [25]:
for s in ["   +0 123", "-5-", "  0000000000000000000000000001", ".1", "-2147483649"]:
    print(f"{s!r:34} -> {sol.myAtoi(s)}")

'   +0 123'                        -> 0
'-5-'                              -> -5
'  0000000000000000000000000001'   -> 1
'.1'                               -> 0
'-2147483649'                      -> -2147483648


## Đáp án Bài 2

In [26]:
def answer_2_skip_spaces(s: str) -> int:
    i = 0
    while i < len(s) and s[i] == " ":
        i += 1
    return i

assert answer_2_skip_spaces("   ab") == 3
assert answer_2_skip_spaces("ab") == 0
assert answer_2_skip_spaces("   ") == 3
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [27]:
def answer_3_my_atoi(s: str) -> int:
    INT_MAX, INT_MIN = 2**31 - 1, -2**31
    n, i = len(s), 0
    while i < n and s[i] == " ":
        i += 1
    sign = 1
    if i < n and s[i] in "+-":
        sign = -1 if s[i] == "-" else 1
        i += 1
    limit = INT_MAX if sign == 1 else INT_MAX + 1
    value = 0
    while i < n and "0" <= s[i] <= "9":
        d = ord(s[i]) - ord("0")
        if value > limit // 10 or (value == limit // 10 and d > limit % 10):
            return INT_MAX if sign == 1 else INT_MIN
        value = value * 10 + d
        i += 1
    return sign * value

assert answer_3_my_atoi("   -042") == -42
assert answer_3_my_atoi("+-12") == 0
assert answer_3_my_atoi("-91283472332") == -2147483648
assert answer_3_my_atoi("") == 0
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [28]:
def answer_4_atoi_bits(s: str, bits: int) -> int:
    MAX, MIN = 2**(bits - 1) - 1, -2**(bits - 1)
    n, i = len(s), 0
    while i < n and s[i] == " ":
        i += 1
    sign = 1
    if i < n and s[i] in "+-":
        sign = -1 if s[i] == "-" else 1
        i += 1
    limit = MAX if sign == 1 else MAX + 1
    value = 0
    while i < n and "0" <= s[i] <= "9":
        d = ord(s[i]) - ord("0")
        if value > limit // 10 or (value == limit // 10 and d > limit % 10):
            return MAX if sign == 1 else MIN
        value = value * 10 + d
        i += 1
    return sign * value

assert answer_4_atoi_bits("200", 8) == 127
assert answer_4_atoi_bits("-128", 8) == -128
assert answer_4_atoi_bits("-129", 8) == -128
assert answer_4_atoi_bits("42", 32) == 42
print("Bài 4 PASS")

Bài 4 PASS

## Đáp án Bài 5

In [29]:
def answer_5_extract_ints(s: str):
    result = []
    n, i = len(s), 0
    while i < n:
        is_digit = "0" <= s[i] <= "9"
        is_neg_start = s[i] == "-" and i + 1 < n and "0" <= s[i + 1] <= "9"
        if not (is_digit or is_neg_start):
            i += 1
            continue
        sign = 1
        if s[i] == "-":
            sign = -1
            i += 1
        value = 0
        while i < n and "0" <= s[i] <= "9":
            value = value * 10 + ord(s[i]) - ord("0")
            i += 1
        result.append(sign * value)
    return result

assert answer_5_extract_ints("a12b-3c--4 5") == [12, -3, -4, 5]
assert answer_5_extract_ints("no digits") == []
assert answer_5_extract_ints("-") == []
print("Bài 5 PASS")

Bài 5 PASS


# 17. Tổng kết

> Đọc đặc tả → tách thành các bước theo thứ tự → một con trỏ `i` đi tới → mỗi bước kiểm tra `i < n` → chống tràn trước khi nhân.

---

## Checklist trước khi nộp

- [ ] Chỉ bỏ dấu cách `' '` ở **đầu**?
- [ ] Chỉ chấp nhận **một** dấu, ngay sau khoảng trắng?
- [ ] Dừng ở ký tự đầu tiên không phải chữ số?
- [ ] Kiểm tra tràn **trước** khi nhân, và **kẹp** (không trả 0)?
- [ ] Giới hạn phía âm là `2^31`?
- [ ] Đã thử `""`, `"   "`, `"-"`, `"+-12"`, `"0-1"`, `"-91283472332"`?